Import Library

> **Image Augmentation**

In [6]:

import tensorflow as tf
to_categorical = tf.keras.utils.to_categorical
array_to_img = tf.keras.preprocessing.image.array_to_img
ReduceLROnPlateau = tf.keras.callbacks.ReduceLROnPlateau
ImageDataGenerator = tf.keras.preprocessing.image.ImageDataGenerator
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objs as go
import plotly.figure_factory as ff
import plotly.express as px
import numpy as np
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix
import pandas as pd

#######################################################################################################

img_rows, img_cols = 224 , 224
batch_size = 32
epochs = 15
num_classes = 50
seed = 1

input_shape = (img_rows, img_cols, 3)
devide_list = tf.test.gpu_device_name()

########################################################################################################

#################################
# Load Dataset From Directory   #
#################################

datagen = ImageDataGenerator(
        rotation_range=0.05,    #Randomly rotate images in the range
        zoom_range = 0.2,       #Randomly zoom image
        width_shift_range=0.1,  #Randomly shift images horizontally
        height_shift_range=0.1, #Randomly shift images vertically
        shear_range=0.05,       #Randomly shear images
        horizontal_flip = True
)

val_test_datagen = ImageDataGenerator(rescale = 1./255)


Load Dataset From Directory

In [8]:
train_it = datagen.flow_from_directory("E:/mystudy/3 term 1/data processing/lab 9/moth/train",
                                       target_size=(img_rows, img_cols),
                                       batch_size=batch_size,
                                       class_mode='categorical',
                                       shuffle=True,
                                       color_mode='rgb',
                                       seed=seed)
test_it = val_test_datagen.flow_from_directory("E:/mystudy/3 term 1/data processing/lab 9/moth/test",
                                               target_size=(img_rows, img_cols),
                                               batch_size=batch_size,
                                               class_mode='categorical',
                                               shuffle=False,
                                               color_mode='rgb',
                                               seed=seed)

val_it = val_test_datagen.flow_from_directory("E:/mystudy/3 term 1/data processing/lab 9/moth/valid",
                                              target_size=(img_rows, img_cols),
                                              batch_size=batch_size,
                                              class_mode='categorical',
                                              shuffle=False,
                                              color_mode='rgb',
                                              seed=seed)

Found 7183 images belonging to 50 classes.
Found 250 images belonging to 50 classes.
Found 250 images belonging to 50 classes.


In [9]:
batchX, batchY = train_it.next()
print('Batch shape=%s, min=%.3f, max=%.3f' % (batchX.shape, batchX.min(), batchX.max()))

Batch shape=(32, 224, 224, 3), min=0.000, max=255.000


In [10]:
labels = list(train_it.class_indices.keys())
print(labels)

['ARCIGERA FLOWER MOTH', 'ATLAS MOTH', 'BANDED TIGER MOTH', 'BIRD CHERRY ERMINE MOTH', 'BLACK RUSTIC MOTH', 'BLAIRS MOCHA', 'BLOTCHED EMERALD MOTH', 'BLUE BORDERED CARPET MOTH', 'CINNABAR MOTH', 'CLEARWING MOTH', 'COMET MOTH', 'DEATHS HEAD HAWK MOTH', 'ELEPHANT HAWK MOTH', 'EMPEROR GUM MOTH', 'EMPEROR MOTH', 'EYED HAWK MOTH', 'FIERY CLEARWING MOTH', 'GARDEN TIGER MOTH', 'HERCULES MOTH', 'HORNET MOTH', 'HUMMING BIRD HAWK MOTH', 'IO MOTH', 'JULY BELLE MOTH', 'KENTISH GLORY MOTH', 'LACE BORDER MOTH', 'LEOPARD MOTH', 'LUNA MOTH', 'MADAGASCAN SUNSET MOTH', 'MAGPIE MOTH', 'MUSLIN MOTH', 'OLEANDER HAWK MOTH', 'OWL MOTH', 'PALPITA VITREALIS MOTH', 'PEACH BLOSSOM MOTH', 'PLUME MOTH', 'POLYPHEMUS MOTH', 'PURPLE BORDERED GOLD MOTH', 'RED NECKED FOOTMAN MOTH', 'REGAL MOTH', 'ROSY MAPLE MOTH', 'ROSY UNDERWING MOTH', 'RUSTY DOT PEARL MOTH', 'SCHORCHED WING MOTH', 'SIXSPOT BURNET MOTH', 'SQUARE SPOT RUSTIC MOTH', 'SUSSEX EMERALD MOTH', 'SWALLOW TAILED MOTH', 'VESTAL MOTH', 'WHITE LINED SPHINX MOTH', 

In [11]:
print(Counter(train_it.classes))

Counter({22: 186, 5: 177, 23: 175, 24: 175, 32: 174, 41: 174, 28: 170, 34: 170, 11: 169, 29: 168, 25: 167, 45: 167, 15: 165, 37: 165, 49: 161, 12: 160, 38: 154, 42: 154, 33: 149, 46: 149, 44: 146, 20: 143, 48: 143, 14: 142, 40: 141, 0: 140, 16: 139, 18: 137, 39: 137, 2: 136, 13: 136, 4: 134, 3: 133, 30: 132, 8: 131, 17: 131, 47: 130, 1: 129, 27: 123, 35: 123, 9: 122, 7: 121, 31: 121, 19: 120, 21: 120, 6: 117, 26: 117, 10: 105, 36: 104, 43: 101})


In [12]:
weights = compute_class_weight('balanced', classes =  np.unique(train_it.classes), y = train_it.classes)
print(weights)

[1.02614286 1.11364341 1.05632353 1.08015038 1.07208955 0.81163842
 1.22786325 1.18727273 1.09664122 1.17754098 1.36819048 0.85005917
 0.897875   1.05632353 1.01169014 0.87066667 1.03352518 1.09664122
 1.04861314 1.19716667 1.00461538 1.19716667 0.77236559 0.82091429
 0.82091429 0.86023952 1.22786325 1.16796748 0.84505882 0.85511905
 1.08833333 1.18727273 0.82563218 0.96416107 0.84505882 1.16796748
 1.38134615 0.87066667 0.93285714 1.04861314 1.01886525 0.82563218
 0.93285714 1.42237624 0.9839726  0.86023952 0.96416107 1.10507692
 1.00461538 0.89229814]


In [13]:
cw = dict(zip(np.unique(train_it.classes), weights))
print(cw)

{0: 1.0261428571428572, 1: 1.1136434108527131, 2: 1.0563235294117648, 3: 1.0801503759398496, 4: 1.0720895522388059, 5: 0.8116384180790961, 6: 1.2278632478632479, 7: 1.1872727272727273, 8: 1.0966412213740457, 9: 1.1775409836065573, 10: 1.3681904761904762, 11: 0.8500591715976331, 12: 0.897875, 13: 1.0563235294117648, 14: 1.0116901408450705, 15: 0.8706666666666667, 16: 1.0335251798561151, 17: 1.0966412213740457, 18: 1.0486131386861315, 19: 1.1971666666666667, 20: 1.0046153846153847, 21: 1.1971666666666667, 22: 0.7723655913978494, 23: 0.8209142857142857, 24: 0.8209142857142857, 25: 0.8602395209580839, 26: 1.2278632478632479, 27: 1.1679674796747967, 28: 0.8450588235294118, 29: 0.8551190476190477, 30: 1.0883333333333334, 31: 1.1872727272727273, 32: 0.8256321839080459, 33: 0.9641610738255033, 34: 0.8450588235294118, 35: 1.1679674796747967, 36: 1.3813461538461538, 37: 0.8706666666666667, 38: 0.9328571428571428, 39: 1.0486131386861315, 40: 1.0188652482269505, 41: 0.8256321839080459, 42: 0.93285

นิยาม Model แบบ Convolutional Neural Network (CNN)

In [14]:
#Feature Extraction
model = tf.keras.Sequential()
model.add(tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

model.add(tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

model.add(tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

model.add(tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

#Image Classification
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(128, activation='relu'))
model.add(tf.keras.layers.Dense(64, activation='relu'))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

In [15]:
STEP_SIZE_TRAIN=train_it.n//train_it.batch_size
STEP_SIZE_VALID=val_it.n//val_it.batch_size
STEP_SIZE_TEST=test_it.n//test_it.batch_size

Compile Model

In [16]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 111, 111, 32)     0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 109, 109, 32)      9248      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 54, 54, 32)       0         
 2D)                                                             
                                                                 
 conv2d_2 (Conv2D)           (None, 52, 52, 64)        18496     
                                                                 
 max_pooling2d_2 (MaxPooling  (None, 26, 26, 64)       0

In [17]:
learning_rate_reduction = ReduceLROnPlateau(monitor='val_loss', patience = 2, verbose=1,factor=0.1, min_lr=0.000001)
callbacks_list = [learning_rate_reduction]

Train Model

In [18]:
his = model.fit(
        train_it,
        validation_data=val_it,
        epochs=epochs,
        class_weight=cw,
        callbacks=callbacks_list,
        shuffle=True,
        steps_per_epoch=STEP_SIZE_TRAIN,
        validation_steps=STEP_SIZE_VALID)

Epoch 1/15
224/224 [==============================] - 157s 695ms/step - loss: 0.4554 - accuracy: 0.9682 - val_loss: 0.4879 - val_accuracy: 0.9800 - lr: 0.0010
Epoch 2/15
224/224 [==============================] - 151s 673ms/step - loss: 0.1012 - accuracy: 0.9800 - val_loss: 0.3614 - val_accuracy: 0.9800 - lr: 0.0010
Epoch 3/15
224/224 [==============================] - 151s 675ms/step - loss: 0.1000 - accuracy: 0.9800 - val_loss: 0.2049 - val_accuracy: 0.9800 - lr: 0.0010
Epoch 4/15
224/224 [==============================] - 151s 673ms/step - loss: 0.0987 - accuracy: 0.9800 - val_loss: 0.1263 - val_accuracy: 0.9800 - lr: 0.0010
Epoch 5/15
224/224 [==============================] - 152s 679ms/step - loss: 0.0982 - accuracy: 0.9800 - val_loss: 0.1127 - val_accuracy: 0.9800 - lr: 0.0010
Epoch 6/15
224/224 [==============================] - 151s 674ms/step - loss: 0.0982 - accuracy: 0.9800 - val_loss: 0.1065 - val_accuracy: 0.9800 - lr: 0.0010
Epoch 7/15
224/224 [==========================

Plot Loss

In [22]:
h1 = go.Scatter(y=his.history['loss'], 
                    mode="lines",
                    line=dict(
                        width=2,
                        color='blue'),
                        name="loss"
                   )
h2 = go.Scatter(y=his.history['val_loss'], 
                    mode="lines",
                    line=dict(
                        width=2,
                        color='red'),
                        name="val_loss"
                   )
                   
data = [h1,h2]
layout1 = go.Layout(title='Loss',
                   xaxis=dict(title='epochs'),
                   yaxis=dict(title=''))
fig1 = go.Figure(data, layout=layout1)
plotly.offline.iplot(fig1)

Plot Accuracy

In [23]:
h1 = go.Scatter(y=his.history['accuracy'], 
                    mode="lines", line=dict(
                    width=2,
                    color='blue'),
                    name="acc"
                   )
h2 = go.Scatter(y=his.history['val_accuracy'], 
                    mode="lines", line=dict(
                    width=2,
                    color='red'),
                    name="val_acc"
                   )

data = [h1,h2]
layout1 = go.Layout(title='Accuracy',
                   xaxis=dict(title='epochs'),
                   yaxis=dict(title=''))
fig1 = go.Figure(data = data, layout=layout1)
plotly.offline.iplot(fig1)

In [24]:
score = model.evaluate(test_it,verbose=0)
print("Test Loss:",score[0])
print("Test Accuracy:",score[1])

Test Loss: 0.09812184423208237
Test Accuracy: 0.9800000190734863
